# Merged Notebook

This notebook will serve the purpose of merging the organized METAR and San Diego On Time report into one usable dataset.

The merge will focus on merging on the variables local_date and local_time from the organized_metar_data.csv file and flight_date and CRS_DEP_TIME from the SAN_OT_report.csv file. In the event that the local_time and CRS_DEP_TIME do not exactly match the closest weather observation prior to CRS_DEP_TIME will be used.

In [1]:
# import libraries
import pandas as pd

In [2]:
# read in the data
wx_data = pd.read_csv("/home/sagemaker-user/ADS-508-Project/clean_data/organized_metar_data.csv")
flt_data = pd.read_csv("/home/sagemaker-user/ADS-508-Project/clean_data/SAN_OT_report.csv")

In [3]:
# create a datetime column for the weather data
wx_data['wx_datetime'] = pd.to_datetime(wx_data['local_date'] + ' ' + wx_data['local_time'], 
                                        format="%m/%d/%y %H:%M:%S", 
                                        errors='coerce')

wx_data[['local_date', 'local_time', 'wx_datetime']].head() 

,local_date,local_time,wx_datetime
0,12/31/22,16:10:03,2022-12-31 16:10:03
1,12/31/22,16:51:03,2022-12-31 16:51:03
2,12/31/22,17:51:03,2022-12-31 17:51:03
3,12/31/22,18:16:03,2022-12-31 18:16:03
4,12/31/22,18:38:03,2022-12-31 18:38:03


In [4]:
# create a datetime column for the flight data
flt_data['flt_datetime'] = pd.to_datetime(flt_data['FL_DATE'] + ' ' + flt_data['CRS_DEP_TIME'], 
                                          format="%Y-%m-%d %H:%M:%S", 
                                          errors='coerce')

flt_data[['FL_DATE', 'CRS_DEP_TIME', 'flt_datetime']].head()

,FL_DATE,CRS_DEP_TIME,flt_datetime
0,2023-01-01,07:15:00,2023-01-01 07:15:00
1,2023-01-01,06:15:00,2023-01-01 06:15:00
2,2023-01-01,22:59:00,2023-01-01 22:59:00
3,2023-01-01,08:15:00,2023-01-01 08:15:00
4,2023-01-01,14:50:00,2023-01-01 14:50:00


In [5]:
# sort the flight data by flt_datetime
flt_data = flt_data.sort_values(by='flt_datetime')

In [6]:
# check both sorted by created datetime fields
print("METAR is monotonic:", wx_data["wx_datetime"].is_monotonic_increasing)
print("Flights are monotonic:", flt_data["flt_datetime"].is_monotonic_increasing)

METAR is monotonic: True
Flights are monotonic: True


In [7]:
# merge the dataframes on the datetime fields created prior
merged_data = pd.merge_asof(flt_data, 
                           wx_data,left_on='flt_datetime',
                           right_on='wx_datetime', 
                           direction='backward')

merged_data.head()

,FL_DATE,CRS_DEP_TIME,DEP_TIME,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,DEP_DEL15,CANCELLED,CANCELLATION_CODE,...,local_time,station_id,wind_dir_degrees,wind_speed_kt,wind_gust_kt,visibility_statute_mi,temperature_c,dewpoint_c,altimeter_hpa,wx_datetime
0,2023-01-01,06:10:00,06:05:00,DL,977,SAN,DTW,0,0,NaN,...,05:51:02,KSAN,290.0,7.0,19.0,6.0,13.3,10.0,29.82,2023-01-01 05:51:02
1,2023-01-01,06:15:00,06:17:00,DL,2423,SAN,SLC,0,0,NaN,...,05:51:02,KSAN,290.0,7.0,19.0,6.0,13.3,10.0,29.82,2023-01-01 05:51:02
2,2023-01-01,06:15:00,06:44:00,AA,1651,SAN,CLT,1,0,NaN,...,05:51:02,KSAN,290.0,7.0,19.0,6.0,13.3,10.0,29.82,2023-01-01 05:51:02
3,2023-01-01,06:15:00,06:15:00,DL,820,SAN,ATL,0,0,NaN,...,05:51:02,KSAN,290.0,7.0,19.0,6.0,13.3,10.0,29.82,2023-01-01 05:51:02
4,2023-01-01,06:15:00,06:13:00,DL,914,SAN,MSP,0,0,NaN,...,05:51:02,KSAN,290.0,7.0,19.0,6.0,13.3,10.0,29.82,2023-01-01 05:51:02


In [8]:
# check the column names
merged_data.columns

Index(['FL_DATE', 'CRS_DEP_TIME', 'DEP_TIME', 'OP_UNIQUE_CARRIER',
       'OP_CARRIER_FL_NUM', 'ORIGIN', 'DEST', 'DEP_DEL15', 'CANCELLED',
       'CANCELLATION_CODE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
       'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'flt_datetime', 'local_date',
       'local_time', 'station_id', 'wind_dir_degrees', 'wind_speed_kt',
       'wind_gust_kt', 'visibility_statute_mi', 'temperature_c', 'dewpoint_c',
       'altimeter_hpa', 'wx_datetime'],
      dtype='object')

In [10]:
# drop unnecessary columns
merged_data = merged_data.drop(columns=['local_date', 'local_time', 'wx_datetime', 'station_id', 'flt_datetime', 'ORIGIN'])

In [11]:
# save the merged data
merged_data.to_csv("/home/sagemaker-user/ADS-508-Project/clean_data/merged_data.csv", index=False)

In [1]:
%%html
<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>